# IVECO Statistics - Main Pipeline VODR

Notebook operativo per generare i report VODR usando widget Databricks e moduli Python versionati.

In [ ]:
from run_local_sample import build_spark
from vodr_config import config_mt_from_vodr
from vodr_pipeline import (
    DEFAULT_VODR_CONFIG,
    DEFAULT_VODR_INPUT_MODE,
    DEFAULT_VODR_OUTPUT_DIR,
    DEFAULT_VODR_SAMPLE_FORMAT,
    DEFAULT_VODR_SAMPLE_PATH,
    parse_config_text,
    run_vodr_pipeline,
)
from engine_utils import log_step

In [ ]:
default_config_text = ','.join(map(str, sorted(DEFAULT_VODR_CONFIG)))

try:
    dbutils
    try:
        dbutils.widgets.get("config")
    except Exception:
        dbutils.widgets.text("config", default_config_text, "Config VODR")
    try:
        dbutils.widgets.get("input_mode")
    except Exception:
        dbutils.widgets.dropdown("input_mode", DEFAULT_VODR_INPUT_MODE, ["fat_table", "sample"], "Input mode")
    try:
        dbutils.widgets.get("join_mission_test")
    except Exception:
        dbutils.widgets.dropdown("join_mission_test", "Yes", ["Yes", "No"], "Join Mission Test")
    try:
        dbutils.widgets.get("join_type")
    except Exception:
        dbutils.widgets.dropdown("join_type", "inner", ["inner", "left"], "Join type")
    try:
        dbutils.widgets.get("export_excel")
    except Exception:
        dbutils.widgets.dropdown("export_excel", "Yes", ["Yes", "No"], "Export Excel")
    try:
        dbutils.widgets.get("output_dir")
    except Exception:
        dbutils.widgets.text("output_dir", DEFAULT_VODR_OUTPUT_DIR, "Output dir")
    try:
        dbutils.widgets.get("sample_path")
    except Exception:
        dbutils.widgets.text("sample_path", DEFAULT_VODR_SAMPLE_PATH, "Sample path")
    try:
        dbutils.widgets.get("input_format")
    except Exception:
        dbutils.widgets.dropdown("input_format", DEFAULT_VODR_SAMPLE_FORMAT, ["parquet", "csv"], "Sample format")

    config_text = dbutils.widgets.get("config")
    input_mode = dbutils.widgets.get("input_mode")
    join_mission_test = dbutils.widgets.get("join_mission_test") == "Yes"
    join_type = dbutils.widgets.get("join_type")
    export_excel = dbutils.widgets.get("export_excel") == "Yes"
    output_dir = dbutils.widgets.get("output_dir")
    sample_path = dbutils.widgets.get("sample_path")
    input_format = dbutils.widgets.get("input_format")
except NameError:
    config_text = default_config_text
    input_mode = DEFAULT_VODR_INPUT_MODE
    join_mission_test = True
    join_type = "inner"
    export_excel = True
    output_dir = DEFAULT_VODR_OUTPUT_DIR
    sample_path = DEFAULT_VODR_SAMPLE_PATH
    input_format = DEFAULT_VODR_SAMPLE_FORMAT

config = parse_config_text(config_text)
log_step(f"Config VODR selezionate: {sorted(config)}")
log_step(f"Config Mission Test collegate: {sorted(config_mt_from_vodr(config))}")
log_step(f"Input mode: {input_mode}")
log_step(f"Join Mission Test: {join_mission_test} ({join_type})")
log_step(f"Export Excel: {export_excel}")
log_step(f"Output dir: {output_dir}")

In [ ]:
try:
    spark
except NameError:
    spark = build_spark("iveco-vodr-pipeline")

vodr_result = run_vodr_pipeline(
    spark=spark,
    config=config,
    input_mode=input_mode,
    sample_path=sample_path,
    input_format=input_format,
    output_dir=output_dir,
    export_excel=export_excel,
    join_mission_test=join_mission_test,
    join_type=join_type,
)

df_vodr = vodr_result["dataframe"]
vodr_sheet_outputs = vodr_result["sheet_outputs"]
vodr_excel_path = vodr_result["excel_path"]
log_step(f"Sheet VODR generati: {[item['sheet_name'] for item in vodr_sheet_outputs]}")

try:
    display(df_vodr.limit(20))
except NameError:
    df_vodr.show(20, truncate=False)

In [ ]:
if vodr_excel_path:
    log_step(f"Excel generato: {vodr_excel_path}")
    try:
        dbutils
        import os

        local_excel_path = os.path.abspath(str(vodr_excel_path))
        dbfs_output_dir = "dbfs:/FileStore/iveco_vodr_output"
        dbfs_excel_path = f"{dbfs_output_dir}/{os.path.basename(local_excel_path)}"

        dbutils.fs.mkdirs(dbfs_output_dir)
        dbutils.fs.cp(f"file:{local_excel_path}", dbfs_excel_path, True)
        log_step(f"Excel copiato su DBFS: {dbfs_excel_path}")

        workspace_url = spark.conf.get("spark.databricks.workspaceUrl", None)
        if workspace_url:
            print(f"Download: https://{workspace_url}/files/iveco_vodr_output/{os.path.basename(local_excel_path)}")
    except NameError:
        pass
else:
    log_step("Nessun Excel generato")